# Pandas Data Engineering Project

This notebook contains practical Data Engineering scenarios using a single business dataset.  
The exercises cover data ingestion, profiling, filtering, cleaning, transformation, aggregation, dates, ranking, reshaping, validation, and ETL-style processing.

**Domain:** Ecommerce_Order_Analytics

**Input file:** `03_Ecommerce_Orders.csv`

**Questions:** 30

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

df = pd.read_csv("03_Ecommerce_Orders.csv")
df.head()


## Dataset Information

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())


## Q1. Numeric summary

Create count, mean, minimum and maximum for the main numeric measure.

In [ ]:
print(df["quantity"].agg(["count","mean","min","max"]))

## Q2. High-value records

Identify records above the 75th percentile of quantity.

In [ ]:
t=    order_id customer_id  ... order_status  order_date
0     O00001       C0158  ...    Delivered  2025-11-02
1     O00002       C0069  ...      Shipped  2025-11-17
2     O00003       C0117  ...    Delivered  2026-07-15
3     O00004       C0151  ...    Delivered  2025-01-01
4     O00005       C0011  ...      Shipped  2025-10-24
..       ...         ...  ...          ...         ...
418   O00419       C0019  ...    Delivered  2025-12-12
419   O00420       C0035  ...      Shipped  2025-04-22
420   O00010       C0150  ...    Delivered  2025-08-10
421   O00041       C0081  ...    Delivered  2025-10-28
422   O00092       C0173  ...    Delivered  2025-12-27

[423 rows x 11 columns]
threshold=df["quantity"].quantile(.75)
print(df[df["quantity"]>threshold].sort_values("quantity",ascending=False).head(20))

## Q3. Category aggregation

Calculate record count, total quantity, and average unit_price for each order_id.

In [ ]:
print(df.groupby("order_id",dropna=False).agg(count=("quantity","count"),total=("quantity","sum"),avg=("unit_price","mean")).sort_values("total",ascending=False))

## Q4. Two-dimensional aggregation

Analyze quantity by order_id and customer_id; keep groups with at least 5 records.

In [ ]:
r=df.groupby(["order_id","customer_id"],dropna=False).agg(count=("quantity","count"),total=("quantity","sum")).query("count>=5").sort_values("total",ascending=False); print(r)

## Q5. Ranking

Rank records by the main numeric measure and display the top 15.

In [ ]:
x=df.copy(); x["rank"]=x["quantity"].rank(method="dense",ascending=False); print(x.sort_values("rank").head(15))

## Q6. Duplicate audit

Identify duplicate records using the first identifier column.

In [ ]:
key="order_id"; print(df[df.duplicated(key,keep=False)].sort_values(key))

## Q7. Duplicate removal

Create a deduplicated dataset using the first identifier column.

In [ ]:
x=df.drop_duplicates("order_id"); print("Before:",len(df),"After:",len(x))

## Q8. Missing-value profile

Calculate null percentage for every column and sort descending.

In [ ]:
print((df.isna().mean()*100).round(2).sort_values(ascending=False))

## Q9. Categorical fill

Fill missing values in a categorical column with Unknown.

In [ ]:
x=df.copy(); x["order_id"]=x["order_id"].fillna("Unknown"); print(x["order_id"].value_counts(dropna=False))

## Q10. Numeric median fill

Fill missing numeric values using the overall median.

In [ ]:
x=df.copy(); x["quantity"]=x["quantity"].fillna(x["quantity"].median()); print(x["quantity"].isna().sum())

## Q11. String search

Find records where order_id contains the letter a, case-insensitively.

In [ ]:
print(df[df["order_id"].astype("string").str.contains("a",case=False,na=False)].head(30))

## Q12. Sorted analysis

Sort the dataset by quantity descending and unit_price ascending.

In [ ]:
print(df.sort_values(["quantity","unit_price"],ascending=[False,True]).head(25))

## Q13. Conditional segmentation

Create a new segment based on quantity using its median as the threshold.

In [ ]:
x=df.copy(); med=x["quantity"].median(); x["segment"]=np.where(x["quantity"]>=med,"Above Median","Below Median"); print(x["segment"].value_counts())

## Q14. Transform

Use groupby transform to calculate each record's group average of the main measure.

In [ ]:
x=df.copy(); x["group_avg"]=x.groupby("order_id")["quantity"].transform("mean"); print(x[["order_id","quantity","group_avg"]].head(25))

## Q15. Group share

Calculate each order_id's share of total quantity.

In [ ]:
g=df.groupby("order_id",dropna=False)["quantity"].sum(); print((g/g.sum()*100).round(2).sort_values(ascending=False))

## Q16. Pivot report

Create a pivot table showing average quantity by order_id and customer_id.

In [ ]:
print(pd.pivot_table(df,index="order_id",columns="customer_id",values="quantity",aggfunc="mean"))

## Q17. Melt report

Create a compact wide summary and convert it to long format using melt.

In [ ]:
w=df.groupby("order_id")[["quantity","unit_price"]].mean().reset_index(); print(w.melt(id_vars="order_id",var_name="metric",value_name="value"))

## Q18. Validation rule

Count records where quantity is negative or missing.

In [ ]:
print("Invalid:",df["quantity"].isna().sum()+(df["quantity"]<0).sum())

## Q19. Outlier detection

Use IQR to identify high outliers in quantity.

In [ ]:
q1=df["quantity"].quantile(.25); q3=df["quantity"].quantile(.75); iqr=q3-q1; print(df[df["quantity"]>q3+1.5*iqr][["quantity"]].head(30))

## Q20. Top group

Find the category in order_id with the highest total quantity.

In [ ]:
print(df.groupby("order_id")["quantity"].sum().nlargest(1))

## Q21. Bottom group

Find the three categories in order_id with the lowest average quantity.

In [ ]:
print(df.groupby("order_id")["quantity"].mean().nsmallest(3))

## Q22. Multiple aggregations

Produce min, max, mean and median of quantity for each order_id.

In [ ]:
print(df.groupby("order_id")["quantity"].agg(["min","max","mean","median"]))

## Q23. Apply rule

Use apply to classify quantity relative to its median.

In [ ]:
x=df.copy(); med=x["quantity"].median(); x["classification"]=x["quantity"].apply(lambda v:"High" if pd.notna(v) and v>=med else "Low"); print(x["classification"].value_counts())

## Q24. Map categories

Create a numeric code for the most frequent categories in the first categorical field.

In [ ]:
codes={{v:i for i,v in enumerate(df["order_id"].dropna().unique())}}; x=df.copy(); x["order_id_code"]=x["order_id"].map(codes); print(x[["order_id","order_id_code"]].head(20))

## Q25. ETL filtering

Build an ETL subset containing non-null quantity, non-null order_id, and values above the median quantity.

In [ ]:
m=df["quantity"].median(); x=df[df["quantity"].notna()&df["order_id"].notna()&(df["quantity"]>m)].copy(); print(x.shape); print(x.head())

## Q26. Chunk processing

Read the input CSV in chunks of 50 rows and calculate the main numeric total.

In [ ]:
total=0
count=0
for chunk in pd.read_csv("03_Ecommerce_Orders.csv",chunksize=50):
    total += chunk["quantity"].sum()
    count += len(chunk)
print("Rows processed:",count)
print("Total:",total)

## Q27. Final report

Create a final order_id-level report with count, total quantity, average unit_price, and rank by total quantity.

In [ ]:
r=df.groupby("order_id",dropna=False).agg(records=("quantity","count"),total=("quantity","sum"),average=("unit_price","mean")).sort_values("total",ascending=False); r["rank"]=r["total"].rank(method="dense",ascending=False).astype(int); print(r)

## End of Project

The notebook is ready to execute with the supplied input CSV.